<a href="https://colab.research.google.com/github/marviiiin/Computer-Vision/blob/main/DL4CV_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 68.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# UCF101 returns clips as (T, H, W, C) tensors
transform = transforms.Compose([
    transforms.Lambda(lambda x: x.permute(0, 3, 1, 2)), # Convert to (T, C, H, W)
    transforms.Lambda(lambda x: x / 255.0),            # Normalize to [0, 1]
    transforms.Resize((224, 224)),                     # Resize frames
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

video_path = "/content/drive/MyDrive/ComputerVision/UCF101horse/videos"
annotation_path = "/content/drive/MyDrive/ComputerVision/UCF101horse/annotations"

ucf101_dataset = datasets.UCF101(
    root=video_path,
    annotation_path=annotation_path,
    frames_per_clip=32,
    step_between_clips=32, # no overlap
    fold=1, # 1, 2, or 3
    train=True,
    transform=transform
)

dataloader = DataLoader(ucf101_dataset, batch_size=8, shuffle=True)

print(f"Dataset size: {len(ucf101_dataset)}")

100%|██████████| 18/18 [00:08<00:00,  2.02it/s]

Dataset size: 1394


In [ ]:
for video, _, label in dataloader:
  print(video.shape)
  print(label)
  break

torch.Size([8, 32, 3, 224, 224])
tensor([0, 1, 1, 1, 1, 1, 1, 1])


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class VisualFeatures(torch.nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights='DEFAULT')
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

    def forward(self, x):
        bs, ss, c, h, w = x.shape
        x = x.view(bs*ss, c, h, w)
        x = self.backbone(x)
        x = x.view(bs, ss, -1)
        return x

class VideoClassifier(torch.nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.rnn = torch.nn.LSTM(input_size, hidden_size, 1, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        #print(x.size())
        x, (y, z) = self.rnn(x)
        #print(x.size(), y.size(), z.size())
        x = x[:,-1]
        #print(x.size())
        x = self.fc(x)
        #print(x.size())
        return x

cnn = VisualFeatures().cuda()
vc = VideoClassifier(512, 512, 2).cuda()

optimizer = torch.optim.Adam(vc.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

for i in range(100):
  vc.train()
  for x, _, y in dataloader:
    with torch.no_grad():
      feat = cnn(x.cuda())

    optimizer.zero_grad()
    outputs = vc(feat)
    loss = criterion(outputs, y.cuda())
    loss.backward()
    optimizer.step()

    y_ = torch.argmax(outputs, dim=1)
    correct = torch.sum((y_ == y.cuda()).float()).item()

    print(loss.item(), correct/len(x))
  break

0.6782486438751221 0.75
0.5418407917022705 0.75
0.5049771666526794 0.5
0.46872884035110474 0.625
0.4886728823184967 0.625
0.8064234256744385 0.625
0.2207430601119995 1.0
0.17223531007766724 1.0
0.418087899684906 0.875
0.35000962018966675 0.875
0.1610998660326004 1.0
0.21470555663108826 1.0
0.1442771553993225 1.0
0.6416534185409546 0.625
0.21923258900642395 0.875
0.3081374764442444 0.875
0.06191549077630043 1.0
0.06387285888195038 1.0
1.0106782913208008 0.625
0.52708500623703 0.75
0.42112720012664795 0.75
0.1351647973060608 1.0
0.11319196224212646 1.0
0.35602229833602905 0.875
0.3028431534767151 0.875
0.202329620718956 0.875
0.6423100829124451 0.875
0.17676886916160583 1.0
0.40130287408828735 0.75
0.2033202350139618 1.0
0.2784103453159332 0.875
0.2874384820461273 0.875
0.16842621564865112 1.0
0.09610098600387573 1.0
0.48415324091911316 0.75
0.11626601964235306 1.0
0.09374679625034332 1.0
0.174167662858963 1.0
0.045213472098112106 1.0
0.04721060022711754 1.0
0.5976767539978027 0.75
0.043